In [1]:
import sys
import os
from pathlib import Path
# Ajout du répertoire parent au path pour les imports source/
_parent = Path().resolve().parent
if str(_parent) not in sys.path:
    sys.path.insert(0, str(_parent))
    
import pandas as pd

import yaml
cfg = yaml.safe_load(Path('../config/config.yaml').read_text())

path_data     = cfg['path_data']
path_fold     = cfg['path_fold']

splits_info = pd.read_csv(path_fold)

from src.evaluation import evaluate_river

df_raw      = pd.read_csv(path_data)

results_river = evaluate_river(splits_info, df_raw)
print(results_river[['MAE','RMSE','MAPE']].mean())

MAE      3.547237
RMSE     4.267192
MAPE    19.643429
dtype: float64


In [4]:
from src.river_model import build_model
from src.evaluation import evaluate_river
import pandas as pd

# avec Adam + régularisation
# model = build_model(lr=0.005, l2=0.001, optimizer="adam")

# grid search simple
for opt in ["sgd", "adam"]:
    for lr in [0.001, 0.01]:  #0.05 explose
        for l2 in [0.0, 0.0001, 0.001, 0.01]:
            print('lr :', lr, ', l2 :', l2, ', opt :', opt)
            results = evaluate_river(splits_info, 
                                     df_raw, 
                                     model_kwargs={"lr": lr, 
                                                   "l2": l2, 
                                                   "optimizer": opt})
            print(results[['MAE','RMSE', 'R2', 'MAPE']].mean())

lr : 0.001 l2 : 0.0001 opt : sgd
MAE      2.749336
RMSE     3.415065
R2       0.000313
MAPE    13.449259
dtype: float64
lr : 0.001 l2 : 0.001 opt : sgd
MAE      2.749257
RMSE     3.415411
R2       0.001483
MAPE    13.445525
dtype: float64
lr : 0.001 l2 : 0.01 opt : sgd
MAE      2.748669
RMSE     3.419118
R2       0.012755
MAPE    13.410431
dtype: float64
lr : 0.01 l2 : 0.0001 opt : sgd
MAE      3.540794
RMSE     4.261389
R2      -1.138089
MAPE    19.600836
dtype: float64
lr : 0.01 l2 : 0.001 opt : sgd
MAE      3.485521
RMSE     4.212404
R2      -1.068090
MAPE    19.233273
dtype: float64
lr : 0.01 l2 : 0.01 opt : sgd
MAE      3.150460
RMSE     3.945339
R2      -0.635087
MAPE    16.754808
dtype: float64
lr : 0.001 l2 : 0.0001 opt : adam
MAE      3.185967
RMSE     3.983166
R2       0.374998
MAPE    14.807036
dtype: float64
lr : 0.001 l2 : 0.001 opt : adam
MAE      3.186122
RMSE     3.983387
R2       0.375063
MAPE    14.806983
dtype: float64
lr : 0.001 l2 : 0.01 opt : adam
MAE      3.18775

In [5]:
best = {"mae": float("inf")}

for l2 in [0.0, 0.0001, 0.001, 0.01]:
    res = evaluate_river(splits_info, 
                         df_raw,
                         model_kwargs={"lr": 0.01, "optimizer": "adam", "l2": l2})
    mae = res["MAE"].mean()
    print(f"l2={l2:.4f}  MAE={mae:.4f}")
    if mae < best["mae"]:
        best = {"mae": mae, "l2": l2}

print(f"\nMeilleur : l2={best['l2']}  MAE={best['mae']:.4f}")

l2=0.0000  MAE=2.7490
l2=0.0001  MAE=2.7490
l2=0.0010  MAE=2.7493
l2=0.0100  MAE=2.7522

Meilleur : l2=0.0  MAE=2.7490
